# Deutsch-Jozsa
One query decides whether an oracle $f:\{0,1\}^n\!\to\!\{0,1\}$ is **constant** or **balanced**.
Measure the $n$-qubit input register: all-zeros $\Rightarrow$ constant, anything else $\Rightarrow$ balanced.
Runs on the local Aer simulator (noiseless).

In [ ]:
# Oracle: 'constant' does nothing to the target; 'balanced' CXs every input into it
from qiskit import QuantumCircuit

def dj_oracle(n: int, kind: str) -> QuantumCircuit:
    oracle = QuantumCircuit(n + 1)
    if kind == "balanced":
        for q in range(n):
            oracle.cx(q, n)
    elif kind != "constant":
        raise ValueError("kind must be 'constant' or 'balanced'")
    return oracle

In [ ]:
# Assemble the full Deutsch-Jozsa circuit and draw it
n = 3
kind = "balanced"   # try "constant" too

qc = QuantumCircuit(n + 1, n)
qc.x(n)                       # target starts in |1>
qc.h(range(n + 1))            # superposition + |-> on target
qc.barrier()
qc.compose(dj_oracle(n, kind), inplace=True)
qc.barrier()
qc.h(range(n))               # interference on the input register
qc.measure(range(n), range(n))

qc.draw("mpl")

In [ ]:
# Run on the Aer simulator
from qiskit_aer import AerSimulator

counts = AerSimulator().run(qc, shots=1024).result().get_counts()

In [ ]:
# All-zeros => constant, anything else => balanced
from qiskit.visualization import plot_histogram

print("most frequent:", max(counts, key=counts.get))
plot_histogram(counts)